# Final Model Evaluation

## Overview
This notebook evaluates the actual production `CatBoost` model on the hold-out dataset. We dive deep into residuals, error distributions, and the model's best and worst predictions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

## 1. Load Model & True Data

In [2]:
X_test = pd.read_parquet('../checkpoints/X_test_cat.parquet')
y_test = np.load('../checkpoints/y_test.npy')

model = CatBoostRegressor()
model.load_model('../models/CatBoost_best_final.cbm')

CATBOOST_CAT = ['season', 'crop', 'state_name', 'crop_season', 'state_crop', 'geo_cluster_10', 'geo_cluster_30', 'geo_cluster_50']
pool = Pool(X_test, cat_features=CATBOOST_CAT)

# Predict and inverse log transform
preds_log = model.predict(pool)
preds = np.expm1(preds_log)

## 2. Final Evaluation Metrics
Confirming the final metrics of the production model.

In [3]:
r2 = r2_score(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)

print(f'Hold-out R²:   {r2:.4f}')
print(f'Hold-out RMSE: {rmse:.4f}')
print(f'Hold-out MAE:  {mae:.4f}')

Hold-out R²:   0.6047
Hold-out RMSE: 10.0143
Hold-out MAE:  1.4211


## 3. Residual Analysis
Plotting Residuals vs. Predicted Values. A healthy model should have residuals clustered randomly around the zero line without obvious patterns.

In [4]:
residuals = y_test - preds
plot_df = pd.DataFrame({'Actual': y_test, 'Predicted': preds, 'Residuals': residuals})

fig = px.scatter(plot_df, x='Predicted', y='Residuals', opacity=0.4, title='Residuals vs. Predicted Values', color_discrete_sequence=['indigo'])
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

## 4. Error Distribution
Plotting the distribution of errors to ensure they are approximately normally distributed around zero.

In [5]:
fig = px.histogram(plot_df, x='Residuals', nbins=100, title='Distribution of Prediction Errors', marginal='box', color_discrete_sequence=['teal'])
fig.show()

## 5. Actual vs Predicted
Visualizing the correlation between actual and predicted yields.

In [6]:
fig = px.scatter(plot_df, x='Actual', y='Predicted', opacity=0.4, title='Actual vs Predicted Yields', color_discrete_sequence=['darkorange'])
fig.add_trace(go.Scatter(x=[0, plot_df['Actual'].max()], y=[0, plot_df['Actual'].max()], mode='lines', line=dict(color='red', dash='dash'), name='Perfect Prediction'))
fig.show()

## 6. Deep Dive: Best and Worst Predictions
Let's extract the rows where the model performed exceptionally well, and where it failed spectacularly.

In [7]:
plot_df['Absolute_Error'] = np.abs(plot_df['Residuals'])
worst_preds = plot_df.sort_values(by='Absolute_Error', ascending=False).head(5)
best_preds = plot_df.sort_values(by='Absolute_Error', ascending=True).head(5)

print("--- Top 5 WORST Predictions ---")
display(worst_preds)
print("\n--- Top 5 BEST Predictions ---")
display(best_preds)

--- Top 5 WORST Predictions ---


,Actual,Predicted,Residuals,Absolute_Error
17577,485.000000,2.009140,482.990860,482.990860
2674,429.666667,1.278694,428.387972,428.387972
23508,393.000000,3.793220,389.206780,389.206780
7498,358.000000,2.344797,355.655203,355.655203
30712,357.000000,5.985271,351.014729,351.014729



--- Top 5 BEST Predictions ---


,Actual,Predicted,Residuals,Absolute_Error
28313,0.611765,0.611754,0.000011,0.000011
4768,0.456604,0.456592,0.000012,0.000012
33578,6.001626,6.001665,-0.000038,0.000038
19475,1.511628,1.511666,-0.000038,0.000038
3855,2.146039,2.145998,0.000041,0.000041
